In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Task 1: Write your code here:
# Read the dataset `Q1_data.csv` using read_csv()

df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# Task 2: Write your code here:
# Inspect the first few rows using head()

df.head()

In [ ]:
# Task 3: Write your code here:
# Display dataset information using info()

df.info()

In [ ]:
# Task 4: Write your code here:
# Show statistical description using describe()

df.describe()

In [ ]:
# Task 5: Write your code here:
# Plot the target distribution (delivery_time)

def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Delivery_Time Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
# Drop the 'Order_ID' column from the data

df2 = df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
# Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )

# Missing values
print("Missing values:")
print(df2.isnull().sum())

stat_cols = ['Weather', 'Traffic_Level', 'Time_of_Day' , 'Courier_Experience_yrs' , 'Delivery_Time']
df_clean = df2.dropna(subset=stat_cols).copy()

In [ ]:
# Task 3: Write your code here:
# Check and remove duplicates if any exist

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
# Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

#  categorical columns
categorical_cols = df_clean.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

# output = Categorical Columns: ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
# we have Categorical Columns, so we need Encode

# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.head()


In [ ]:
# Task 5: Write your code here:
# Apply feature scaling for all features (Use StandardScaler)


numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
# Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

# All good here

In [ ]:
# Task 1: Write your code here:
# Split the dataset into features (X) and target (y)
from sklearn.model_selection import train_test_split, KFold


X = df_clean.drop("Delivery_Time", axis=1).astype(float)
y = df_clean['Delivery_Time'].astype(float)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Scale features - fit on train, transform both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train_scaled):
    X_fold_train, X_fold_val = X_train_scaled[train_idx], X_train_scaled[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE:  ${mae_scores.mean():,.2f}")

In [ ]:
# Gather importances from the models (from the last fold)
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

feature_cols = ["Distance_km" ,"Weather" ,"Traffic_Level" ,"Time_of_Day" ,"Vehicle_Type","Preparation_Time_min" ,"Courier_Experience_yrs" ]


# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(y_fold_pred, bins=50, edgecolor='black')
plt.title('predicted delivery time ')
plt.xlabel('delivery time')
plt.ylabel('predicted delivery time ')
plt.show()

In [ ]:
!pip install CatBoost

In [ ]:
# Task Bonus: Write your code here:

from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor


models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)}

In [ ]:
# Storage for results
all_results = {}

for name in models:
  all_results[name] = {'mse': [], 'rmse': [], 'r2': []}

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = sklearn_mse(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)

    # Store results
    all_results[model_name]["mse"].append(mse)
    all_results[model_name]["rmse"].append(rmse)
    all_results[model_name]["r2"].append(r2)

In [ ]:
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  MSE:  {np.mean(all_results[model_name]['mse']):.4f}")
  print(f"  RMSE: {np.mean(all_results[model_name]['rmse']):.4f}")
  print(f"  R2:    {np.mean(all_results[model_name]['r2']):.4f}")